# DynamicBind 虚拟筛选教程

**论文**: Lu Wei, et al. *DynamicBind: predicting ligand-specific protein-ligand complex structure with a deep equivariant generative model.* Nature Communications, 2024.

## 核心创新

DynamicBind 在 DiffDock-Pocket 的基础上，将扩散自由度从 4 维扩展到 **6 维**，同时建模配体和蛋白残基的完整运动自由度:

| 方法 | 扩散自由度 | 说明 |
|------|-----------|------|
| DiffDock | 3-way | 配体平移 + 旋转 + 扭转 |
| DiffDock-Pocket | 4-way | 配体 3-way + 侧链 chi 角 |
| **DynamicBind** | **6-way** | 配体 3-way + **残基平移** + **残基旋转** + **残基 chi 角** |

DynamicBind 的关键设计:
1. **蛋白残基不再固定** -- 每个残基拥有独立的平移、旋转和侧链扭转自由度
2. **差异化噪声调度** -- 残基使用 power-law 调度 (更保守)，配体使用 exponential 调度
3. **自适应蛋白动力学** -- 残基更新仅在配体接近口袋时激活
4. **局部坐标系表示** -- 残基采用 N-CA-C 三点定义的局部坐标系

## 学习目标

1. 理解 6-way 扩散模型的设计动机与物理直觉
2. 掌握蛋白残基运动的三个自由度 (平移、旋转、侧链扭转)
3. 理解配体与残基使用不同噪声调度的原因
4. 实现简化版 DynamicBind 并在 CASF-2016 数据集上进行对接实验

In [ ]:
# ============================================================
# 环境与路径设置
# ============================================================
import sys
import os
import math
import warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger
from scipy.spatial import distance_matrix
import matplotlib.pyplot as plt
import pandas as pd

# BioPython 用于残基级别的蛋白解析
from Bio.PDB import PDBParser

%matplotlib inline

# 抑制 RDKit / BioPython 的冗余警告
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')


def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    from pathlib import Path
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录:   {DATA_DIR}")

# 版本信息
version_info = pd.DataFrame({
    '库': ['Python', 'PyTorch', 'NumPy', 'RDKit', 'SciPy', 'BioPython'],
    '版本': [
        sys.version.split()[0],
        torch.__version__,
        np.__version__,
        Chem.rdBase.rdkitVersion,
        __import__('scipy').__version__,
        __import__('Bio').__version__,
    ]
})
display(version_info)

## 1. 超参数设置

DynamicBind 使用 6 个独立的噪声通道，其中配体和残基使用不同的噪声调度策略:

- **配体 (exponential 调度)**: $\sigma(t) = \sigma_{\max}^t$，噪声范围大 (平移可达 10 Angstrom)
- **残基 (power-law 调度)**: $\sigma(t) = \sigma_{\min} + (\sigma_{\max} - \sigma_{\min}) \cdot (5t)^{0.3}$，噪声范围小 (平移最多 1 Angstrom)

这种差异反映了物理直觉: 蛋白骨架运动受到更多约束，构象变化幅度远小于配体位姿变化。

In [ ]:
# ============================================================
# 超参数定义
# ============================================================

# ---------- 模型与训练参数 ----------
ATOM_FEAT_DIM = 10          # 配体原子特征维度
RES_FEAT_DIM = 21           # 残基特征维度 (20 种标准氨基酸 + 1 other)
HIDDEN_DIM = 128            # 隐层维度
N_INTERACTION_LAYERS = 4    # 蛋白-配体交互层数
DISTANCE_CUTOFF = 8.0       # 残基-原子交互距离阈值 (Angstrom)
N_EPOCHS = 200              # 训练轮数
LR = 1e-4                   # 学习率 (扩散模型使用较小学习率)
BATCH_SIZE = 1              # 变长图，逐样本处理
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- 配体扩散噪声参数 (exponential 调度) ----------
# 配体自由度使用与 DiffDock 相同的指数调度: sigma(t) = sigma_max^t
SIGMA_TR_MAX = 10.0         # 配体平移噪声最大标准差 (Angstrom)
SIGMA_ROT_MAX = 1.5         # 配体旋转噪声最大标准差 (弧度)
SIGMA_TOR_MAX = np.pi       # 配体扭转角噪声最大标准差

# ---------- 残基扩散噪声参数 (power-law 调度) ----------
# DynamicBind 的关键设计: 残基使用更保守的噪声范围和不同的调度函数
# 因为蛋白质构象变化幅度远小于配体位姿变化
SIGMA_RES_TR_MIN = 0.01     # 残基平移噪声最小标准差 (Angstrom)
SIGMA_RES_TR_MAX = 1.0      # 残基平移噪声最大标准差 (Angstrom)
SIGMA_RES_ROT_MIN = 0.01    # 残基旋转噪声最小标准差 (弧度)
SIGMA_RES_ROT_MAX = 1.0     # 残基旋转噪声最大标准差 (弧度)
SIGMA_RES_CHI_MIN = 0.01    # 残基侧链 chi 噪声最小标准差 (弧度)
SIGMA_RES_CHI_MAX = 1.0     # 残基侧链 chi 噪声最大标准差 (弧度)

torch.manual_seed(SEED)
np.random.seed(SEED)

# 显示超参数
params_df = pd.DataFrame({
    '参数': [
        'Device', '扩散模式',
        '配体平移噪声 sigma_max', '配体旋转噪声 sigma_max', '配体扭转噪声 sigma_max',
        '残基平移噪声 [min, max]', '残基旋转噪声 [min, max]', '残基 chi 噪声 [min, max]',
        '隐层维度', '交互层数', '距离阈值', '训练轮数', '学习率', '批大小'
    ],
    '值': [
        str(DEVICE), '6-way (lig tr+rot+tor, res tr+rot+chi)',
        f'{SIGMA_TR_MAX} A', f'{SIGMA_ROT_MAX} rad', f'{SIGMA_TOR_MAX:.4f} rad',
        f'[{SIGMA_RES_TR_MIN}, {SIGMA_RES_TR_MAX}] A',
        f'[{SIGMA_RES_ROT_MIN}, {SIGMA_RES_ROT_MAX}] rad',
        f'[{SIGMA_RES_CHI_MIN}, {SIGMA_RES_CHI_MAX}] rad',
        str(HIDDEN_DIM), str(N_INTERACTION_LAYERS),
        f'{DISTANCE_CUTOFF} A', str(N_EPOCHS), str(LR), str(BATCH_SIZE)
    ]
})
display(params_df)

## 2. 数据加载与特征提取

DynamicBind 需要提取以下信息:

**配体**: 原子级别特征 (元素类型 one-hot + 芳香性) 和 3D 坐标

**蛋白残基**: 残基类型 one-hot、CA/CB/N/C 原子坐标 (用于定义局部坐标系)、侧链原子坐标

其中 N-CA-C 三点定义残基的局部坐标系:
- **N**: 氮原子 (参考点)
- **CA**: alpha 碳 (原点)
- **C**: 羰基碳 (参考点)

此外还需要:
- **chi1 角度**: 从 CA->CB 方向向量计算的侧链扭转角代理值
- **扩散噪声工具函数**: 6-way 噪声调度、加噪函数、Rodrigues 旋转公式等

In [ ]:
# ============================================================
# 特征提取与扩散工具函数
# ============================================================

# ---- 标准氨基酸列表 ----
STANDARD_AAS = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
                'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']

# ---- 元素符号 -> one-hot 映射 (9类 + 1 芳香性 = 10维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']


def parse_coreset(path):
    """解析 CoreSet.dat，返回 {pdbid: logKa} 字典。跳过以 '#' 开头的注释行。"""
    labels = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbid = parts[0]
            logKa = float(parts[3])
            labels[pdbid] = logKa
    print(f"[INFO] 从 CoreSet.dat 读取到 {len(labels)} 个复合物标签")
    return labels


def atom_features(atom):
    """
    构建 10 维配体原子特征向量:
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0
    feat[9] = float(atom.GetIsAromatic())
    return feat


def residue_features(resname):
    """
    构建 21 维残基特征向量:
      - 前 20 维: 标准氨基酸类型 one-hot
      - 第 21 维: 非标准氨基酸 (other)
    """
    feat = np.zeros(RES_FEAT_DIM, dtype=np.float32)
    if resname in STANDARD_AAS:
        feat[STANDARD_AAS.index(resname)] = 1.0
    else:
        feat[20] = 1.0
    return feat


def extract_residue_data(pocket_pdb):
    """
    从口袋 PDB 文件中提取残基级别信息 (用 BioPython PDBParser):
      - res_feats  : (N_r, 21)  残基类型特征
      - ca_coords  : (N_r, 3)   CA 原子坐标
      - cb_coords  : (N_r, 3)   CB 原子坐标 (Gly 无 CB 时用 CA 代替)
      - n_coords   : (N_r, 3)   N 原子坐标  (局部坐标系定义点之一)
      - c_coords   : (N_r, 3)   C 原子坐标  (局部坐标系定义点之一)
      - sc_atoms   : list        每个残基的侧链原子坐标列表

    DynamicBind 使用 N-CA-C 三点定义残基的局部坐标系:
      - N:  氮原子 (局部坐标系的参考点)
      - CA: alpha 碳 (局部坐标系的原点)
      - C:  羰基碳 (局部坐标系的参考点)
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pocket', pocket_pdb)

    res_feats_list = []
    ca_coords_list = []
    cb_coords_list = []
    n_coords_list = []
    c_coords_list = []
    sc_atoms_list = []

    # 主链原子名 (排除这些以获得侧链原子)
    backbone_atoms = {'N', 'CA', 'C', 'O', 'OXT'}

    for model in structure:
        for chain in model:
            for residue in chain:
                # 跳过水分子和 HETATM
                if residue.id[0] != ' ':
                    continue
                resname = residue.get_resname().strip()

                # 需要 N, CA, C 三个原子来定义局部坐标系
                if 'CA' not in residue or 'N' not in residue or 'C' not in residue:
                    continue
                ca_coord = residue['CA'].get_vector().get_array()
                n_coord = residue['N'].get_vector().get_array()
                c_coord = residue['C'].get_vector().get_array()

                # CB 坐标 (Gly 没有 CB，用 CA 代替)
                if 'CB' in residue:
                    cb_coord = residue['CB'].get_vector().get_array()
                else:
                    cb_coord = ca_coord.copy()

                # 收集侧链原子坐标
                sc_coords = []
                for atom in residue:
                    if atom.get_name() not in backbone_atoms:
                        sc_coords.append(atom.get_vector().get_array())

                res_feats_list.append(residue_features(resname))
                ca_coords_list.append(ca_coord)
                cb_coords_list.append(cb_coord)
                n_coords_list.append(n_coord)
                c_coords_list.append(c_coord)
                sc_atoms_list.append(
                    np.array(sc_coords, dtype=np.float32) if len(sc_coords) > 0
                    else np.zeros((0, 3), dtype=np.float32)
                )
        break  # 只取第一个 model

    if len(ca_coords_list) == 0:
        return None

    res_feats = np.array(res_feats_list, dtype=np.float32)
    ca_coords = np.array(ca_coords_list, dtype=np.float32)
    cb_coords = np.array(cb_coords_list, dtype=np.float32)
    n_coords = np.array(n_coords_list, dtype=np.float32)
    c_coords = np.array(c_coords_list, dtype=np.float32)
    return res_feats, ca_coords, cb_coords, n_coords, c_coords, sc_atoms_list


def extract_chi1_angles(ca_coords, cb_coords):
    """
    计算简化的 chi1 角度代理值。
    用 CA->CB 方向向量在 xy 平面上的投影角作为 chi1 的代理。
    对于 Gly (CB == CA)，chi1 设为 0。

    返回: chi1_angles (N_r,) 弧度制
    """
    diff = cb_coords - ca_coords                     # (N_r, 3)
    norms = np.linalg.norm(diff, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-6, None)
    direction = diff / norms
    chi1 = np.arctan2(direction[:, 1], direction[:, 0]).astype(np.float32)
    return chi1


def axis_angle_to_matrix(axis_angle):
    """
    Rodrigues 公式: 轴角向量 (3,) -> 旋转矩阵 (3,3)。
    方向为旋转轴，模长为旋转角度 (弧度)。
    """
    axis_angle = np.asarray(axis_angle, dtype=np.float64)
    angle = np.linalg.norm(axis_angle)
    if angle < 1e-8:
        return np.eye(3, dtype=np.float32)
    k = axis_angle / angle
    K = np.array([[0, -k[2], k[1]],
                  [k[2], 0, -k[0]],
                  [-k[1], k[0], 0]])
    R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)
    return R.astype(np.float32)


def modify_sidechain_torsion(sc_atoms, ca_coord, cb_coord, chi1_update):
    """
    通过绕 CA->CB 轴旋转来模拟侧链扭转角的变化 (Rodrigues 旋转公式)。

    参数:
      sc_atoms    : (N_sc, 3)  侧链原子坐标
      ca_coord    : (3,)       CA 原子坐标 (旋转轴起点)
      cb_coord    : (3,)       CB 原子坐标 (旋转轴方向)
      chi1_update : float      旋转角度 (弧度)

    返回: new_sc_atoms : (N_sc, 3)
    """
    if len(sc_atoms) == 0:
        return sc_atoms.copy()

    axis = cb_coord - ca_coord
    axis_norm = np.linalg.norm(axis)
    if axis_norm < 1e-6:
        return sc_atoms.copy()          # Gly 无侧链轴
    axis = axis / axis_norm

    cos_a = np.cos(chi1_update)
    sin_a = np.sin(chi1_update)
    centered = sc_atoms - ca_coord
    cross = np.cross(axis, centered)
    dot = np.dot(centered, axis).reshape(-1, 1)
    rotated = centered * cos_a + cross * sin_a + axis * dot * (1 - cos_a)
    return rotated + ca_coord


def load_ligand(pdbid):
    """加载配体分子，返回 (原子特征, 3D 坐标)。优先 mol2，备选 sdf。"""
    ligand_mol2 = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_ligand.mol2")
    ligand_sdf = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_ligand.sdf")

    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        supplier = Chem.SDMolSupplier(ligand_sdf, sanitize=False)
        for m in supplier:
            if m is not None:
                lig_mol = m
                break
    if lig_mol is None:
        return None

    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass
    if lig_mol.GetNumAtoms() == 0:
        return None

    conf = lig_mol.GetConformer()
    coords = np.array([conf.GetAtomPosition(i) for i in range(lig_mol.GetNumAtoms())],
                       dtype=np.float32)
    feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)
    return feats, coords


# ---- 正弦位置编码 (扩散时间步嵌入) ----
class SinusoidalEmbedding(nn.Module):
    """
    正弦/余弦位置编码，将标量时间步 t 映射到高维向量。
    对应原始代码 diffusion_utils.py 中的 sinusoidal_embedding 函数。
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        """t: (B,) -> (B, dim)"""
        if t.dim() == 0:
            t = t.unsqueeze(0)
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000.0)
            * torch.arange(half, device=t.device, dtype=torch.float32) / half
        )
        args = t.unsqueeze(-1) * freqs.unsqueeze(0)
        return torch.cat([args.sin(), args.cos()], dim=-1)


# ---- 6-way 噪声调度 ----
def t_to_sigma_6way(t):
    """
    将归一化时间 t in [0,1] 映射到 6 个独立的噪声标准差。

    DynamicBind 的核心创新之一: 配体和残基使用不同的噪声调度函数!

    配体 (exponential 调度): sigma(t) = sigma_max^t
      - 与 DiffDock / DiffDock-Pocket 相同
      - 噪声范围大 (平移可达 10 Angstrom)

    残基 (power-law 调度):
      sigma(t) = clamp(sigma_min + (sigma_max - sigma_min) * (t*5)^0.3, max=1.0)
      - 更保守的噪声范围 (平移最多 1 Angstrom)
      - 缓慢增长的噪声 -- 蛋白构象变化幅度远小于配体位姿变化
      - 这体现了物理直觉: 蛋白骨架运动受到更多约束

    返回: (sigma_tr, sigma_rot, sigma_tor, sigma_res_tr, sigma_res_rot, sigma_res_chi)
    """
    # 配体: exponential 调度
    sigma_tr = SIGMA_TR_MAX ** t
    sigma_rot = SIGMA_ROT_MAX ** t
    sigma_tor = SIGMA_TOR_MAX ** t

    # 残基: power-law 调度 (来自 DynamicBind 原始实现)
    sigma_res_tr = min(
        SIGMA_RES_TR_MIN + (SIGMA_RES_TR_MAX - SIGMA_RES_TR_MIN) * (t * 5.0) ** 0.3,
        1.0
    )
    sigma_res_rot = min(
        SIGMA_RES_ROT_MIN + (SIGMA_RES_ROT_MAX - SIGMA_RES_ROT_MIN) * (t * 5.0) ** 0.3,
        1.0
    )
    sigma_res_chi = min(
        SIGMA_RES_CHI_MIN + (SIGMA_RES_CHI_MAX - SIGMA_RES_CHI_MIN) * (t * 5.0) ** 0.3,
        1.0
    )

    return sigma_tr, sigma_rot, sigma_tor, sigma_res_tr, sigma_res_rot, sigma_res_chi


def apply_noise_to_ligand(lig_coords, t):
    """
    对配体坐标施加 3-way 噪声 (平移 + 旋转 + 扭转)。
    与 DiffDock / DiffDock-Pocket 相同的配体加噪过程。

    返回: (noisy_coords, noise_tr, noise_rot, noise_tor)
    """
    sigma_tr, sigma_rot, sigma_tor, _, _, _ = t_to_sigma_6way(t)

    # ---- 1. 平移噪声 ----
    noise_tr = np.random.randn(3).astype(np.float32) * sigma_tr
    noisy = lig_coords + noise_tr

    # ---- 2. 旋转噪声 (绕质心随机旋转) ----
    center = noisy.mean(axis=0)
    axis = np.random.randn(3).astype(np.float32)
    axis_norm = np.linalg.norm(axis)
    if axis_norm > 1e-6:
        axis = axis / axis_norm
    else:
        axis = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    angle = np.random.randn() * sigma_rot
    noise_rot = axis * angle  # 轴角表示 (3,)

    cos_a, sin_a = np.cos(angle), np.sin(angle)
    centered = noisy - center
    cross = np.cross(axis, centered)
    dot = np.dot(centered, axis).reshape(-1, 1)
    noisy = centered * cos_a + cross * sin_a + axis * dot * (1 - cos_a) + center

    # ---- 3. 扭转角噪声 (简化: 径向扰动) ----
    noise_tor_val = np.random.randn() * sigma_tor
    radial = noisy - center
    radial_norms = np.linalg.norm(radial, axis=1, keepdims=True)
    radial_norms = np.clip(radial_norms, 1e-6, None)
    noisy = noisy + (radial / radial_norms) * noise_tor_val * 0.1

    return noisy, noise_tr, noise_rot, np.float32(noise_tor_val)


def apply_noise_to_residues(ca_coords, n_coords, c_coords, chi1_angles, t):
    """
    对蛋白残基施加 3-way 噪声 (平移 + 旋转 + chi角)。
    这是 DynamicBind 相比 DiffDock-Pocket 的核心新增!

    DiffDock-Pocket 只扰动侧链 chi 角 (1 维)，
    DynamicBind 额外扰动每个残基的:
      - 平移: CA 位置偏移 (模拟骨架运动)
      - 旋转: 局部坐标系旋转 (模拟骨架朝向变化)
      - chi 角: 侧链扭转 (与 DiffDock-Pocket 相同)

    参数:
      ca_coords  : (N_r, 3)  CA 原子坐标
      n_coords   : (N_r, 3)  N 原子坐标
      c_coords   : (N_r, 3)  C 原子坐标
      chi1_angles: (N_r,)    侧链 chi1 角度
      t          : float     归一化时间步

    返回:
      noisy_ca, noisy_n, noisy_c, noisy_chi1,
      noise_res_tr, noise_res_rot, noise_res_chi
    """
    _, _, _, sigma_res_tr, sigma_res_rot, sigma_res_chi = t_to_sigma_6way(t)
    n_res = len(ca_coords)

    # ---- 1. 残基平移噪声: 每个残基独立平移 ----
    # 对 CA 位置加高斯噪声，同时平移 N 和 C 原子 (刚体平移)
    noise_res_tr = np.random.randn(n_res, 3).astype(np.float32) * sigma_res_tr
    noisy_ca = ca_coords + noise_res_tr
    noisy_n = n_coords + noise_res_tr
    noisy_c = c_coords + noise_res_tr

    # ---- 2. 残基旋转噪声: 绕 CA 轴旋转局部坐标系 ----
    # 每个残基独立旋转，以 CA 为支点旋转 N 和 C 原子
    noise_res_rot = np.random.randn(n_res, 3).astype(np.float32) * sigma_res_rot
    for i in range(n_res):
        rot_mat = axis_angle_to_matrix(noise_res_rot[i])
        # 以 CA 为支点旋转 N 和 C
        noisy_n[i] = rot_mat @ (noisy_n[i] - noisy_ca[i]) + noisy_ca[i]
        noisy_c[i] = rot_mat @ (noisy_c[i] - noisy_ca[i]) + noisy_ca[i]

    # ---- 3. 侧链 chi 角噪声 ----
    noise_res_chi = np.random.randn(n_res).astype(np.float32) * sigma_res_chi
    noisy_chi1 = chi1_angles + noise_res_chi
    noisy_chi1 = np.arctan2(np.sin(noisy_chi1), np.cos(noisy_chi1))  # 归一化到 [-pi, pi]

    return noisy_ca, noisy_n, noisy_c, noisy_chi1, noise_res_tr, noise_res_rot, noise_res_chi


def compute_rmsd(pred, true):
    """计算 RMSD (均方根偏差)。"""
    return np.sqrt(np.mean(np.sum((pred - true) ** 2, axis=1)))

In [ ]:
# ============================================================
# 展示一个样本的数据
# ============================================================

labels = parse_coreset(CORESET_FILE)

# 取第一个 PDB 进行展示
sample_pdbid = sorted(labels.keys())[0]
sample_data = build_complex_data(sample_pdbid)

if sample_data is not None:
    print(f"\n样本 PDB ID: {sample_pdbid}")
    sample_info = pd.DataFrame({
        '数据项': [
            '配体原子特征 (lig_feats)',
            '配体坐标 (lig_coords)',
            '残基特征 (res_feats)',
            'CA 坐标 (ca_coords)',
            'CB 坐标 (cb_coords)',
            'N 坐标 (n_coords)',
            'C 坐标 (c_coords)',
            'Chi1 角度 (chi1_angles)',
            '侧链原子列表 (sc_atoms)',
        ],
        '形状': [
            str(sample_data['lig_feats'].shape),
            str(sample_data['lig_coords'].shape),
            str(sample_data['res_feats'].shape),
            str(sample_data['ca_coords'].shape),
            str(sample_data['cb_coords'].shape),
            str(sample_data['n_coords'].shape),
            str(sample_data['c_coords'].shape),
            str(sample_data['chi1_angles'].shape),
            f'{len(sample_data["sc_atoms"])} 个残基的侧链原子',
        ]
    })
    display(sample_info)
else:
    print(f"样本 {sample_pdbid} 解析失败")

## 3. 数据集与数据加载器

将所有复合物数据构建为 PyTorch Dataset，并进行 80/20 训练集/测试集划分。

每个样本包含:
- 配体原子特征与坐标 (真实位姿)
- 蛋白残基特征、CA/CB/N/C 坐标 (局部坐标系)
- 侧链 chi1 角度、侧链原子坐标
- 结合亲和力标签 (logKa)

In [ ]:
# ============================================================
# Dataset 与 DataLoader
# ============================================================

class DynamicBindDataset(Dataset):
    """
    DynamicBind 数据集。每个样本包含:
      - 配体原子特征与坐标 (真实位姿)
      - 蛋白残基特征、CA/CB/N/C 坐标 (局部坐标系)
      - 侧链 chi1 角度
      - 侧链原子坐标
      - PDB ID
    """
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def build_complex_data(pdbid):
    """
    为单个蛋白-配体复合物构建数据。
    与 DiffDock-Pocket 相比新增: N/C 原子坐标 (用于残基局部坐标系)。
    返回字典或 None (解析失败时)。
    """
    pocket_path = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_pocket.pdb")

    # ---- 加载配体 ----
    lig_result = load_ligand(pdbid)
    if lig_result is None:
        return None
    lig_feats, lig_coords = lig_result

    # ---- 加载蛋白口袋 (残基级别) ----
    if not os.path.exists(pocket_path):
        return None
    res_result = extract_residue_data(pocket_path)
    if res_result is None:
        return None
    res_feats, ca_coords, cb_coords, n_coords, c_coords, sc_atoms = res_result

    if len(ca_coords) == 0:
        return None

    # ---- 计算 chi1 角度 ----
    chi1_angles = extract_chi1_angles(ca_coords, cb_coords)

    return {
        'lig_feats': lig_feats,       # (N_l, 10)
        'lig_coords': lig_coords,     # (N_l, 3)  真实配体位姿
        'res_feats': res_feats,       # (N_r, 21)
        'ca_coords': ca_coords,       # (N_r, 3)  CA 原子坐标
        'cb_coords': cb_coords,       # (N_r, 3)  CB 原子坐标
        'n_coords': n_coords,         # (N_r, 3)  N 原子坐标 (局部坐标系)
        'c_coords': c_coords,         # (N_r, 3)  C 原子坐标 (局部坐标系)
        'chi1_angles': chi1_angles,   # (N_r,)
        'sc_atoms': sc_atoms,         # list of (N_sc_i, 3)
    }


# ---- 预处理所有复合物 ----
print("正在构建 DynamicBind 数据...")

all_data = []
failed = 0
for pdbid, logKa in sorted(labels.items()):
    result = build_complex_data(pdbid)
    if result is None:
        failed += 1
        continue
    result['label'] = logKa
    result['pdbid'] = pdbid
    all_data.append(result)

print(f"成功加载 {len(all_data)} 个复合物, 失败 {failed} 个")

# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]

print(f"训练集: {len(train_data)}, 测试集: {len(test_data)}")

train_loader = DataLoader(DynamicBindDataset(train_data), batch_size=BATCH_SIZE,
                           shuffle=True, collate_fn=lambda x: x[0])
test_loader = DataLoader(DynamicBindDataset(test_data), batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=lambda x: x[0])

# 显示数据集统计
dataset_info = pd.DataFrame({
    '数据集': ['训练集', '测试集', '总计'],
    '样本数': [len(train_data), len(test_data), len(all_data)],
    '解析失败': ['', '', str(failed)]
})
display(dataset_info)

## 4. 模型架构

DynamicBind 的分数模型包含 **6 个输出头**，相比 DiffDock-Pocket (4 个头) 新增了残基平移和残基旋转两个头:

| 方法 | 输出头 | 说明 |
|------|--------|------|
| DiffDock | tr + rot + tor | 3 个全局头 (配体) |
| DiffDock-Pocket | tr + rot + tor + sc_tor | 3 个全局头 + 1 个逐残基头 |
| **DynamicBind** | tr + rot + tor + **res_tr** + **res_rot** + **res_chi** | 3 个全局头 + **3 个逐残基头** |

- 配体的 3 个头预测**全局分数** (整个配体的平移/旋转/扭转)
- 残基的 3 个头预测**逐残基分数** (每个残基独立的平移/旋转/chi角)

模型主干使用双向消息传递的蛋白-配体交互层 (InteractionLayer)，通过 4 层交互实现蛋白残基与配体原子之间的信息交换。

In [ ]:
# ============================================================
# 模型架构 -- InteractionLayer
# ============================================================

class InteractionLayer(nn.Module):
    """
    蛋白-配体交互层: 双向消息传递。
    残基级别蛋白表示与原子级别配体表示之间的信息交换。
    与 DiffDock-Pocket 使用相同的交互层设计。
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # 蛋白残基 -> 配体原子
        self.prot_to_lig = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        # 配体原子 -> 蛋白残基
        self.lig_to_prot = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        # 节点更新 MLP
        self.prot_update = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.lig_update = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.layer_norm_prot = nn.LayerNorm(hidden_dim)
        self.layer_norm_lig = nn.LayerNorm(hidden_dim)

    def forward(self, prot_h, lig_h, edge_index, edge_dist):
        """
        prot_h     : (N_r, hidden)   lig_h      : (N_l, hidden)
        edge_index : (2, E) [0]=lig   edge_dist  : (E, 1)
        """
        lig_idx = edge_index[0]
        prot_idx = edge_index[1]

        # ---- 蛋白 -> 配体 消息 ----
        msg_p2l = self.prot_to_lig(
            torch.cat([lig_h[lig_idx], prot_h[prot_idx], edge_dist], dim=-1)
        )
        lig_agg = torch.zeros_like(lig_h)
        count_l = torch.zeros(lig_h.shape[0], 1, device=lig_h.device)
        lig_agg.scatter_add_(0, lig_idx.unsqueeze(-1).expand_as(msg_p2l), msg_p2l)
        count_l.scatter_add_(0, lig_idx.unsqueeze(-1), torch.ones_like(edge_dist))
        lig_agg = lig_agg / count_l.clamp(min=1)

        # ---- 配体 -> 蛋白 消息 ----
        msg_l2p = self.lig_to_prot(
            torch.cat([prot_h[prot_idx], lig_h[lig_idx], edge_dist], dim=-1)
        )
        prot_agg = torch.zeros_like(prot_h)
        count_p = torch.zeros(prot_h.shape[0], 1, device=prot_h.device)
        prot_agg.scatter_add_(0, prot_idx.unsqueeze(-1).expand_as(msg_l2p), msg_l2p)
        count_p.scatter_add_(0, prot_idx.unsqueeze(-1), torch.ones_like(edge_dist))
        prot_agg = prot_agg / count_p.clamp(min=1)

        # ---- 残差更新 ----
        lig_h = self.layer_norm_lig(
            lig_h + self.lig_update(torch.cat([lig_h, lig_agg], dim=-1))
        )
        prot_h = self.layer_norm_prot(
            prot_h + self.prot_update(torch.cat([prot_h, prot_agg], dim=-1))
        )
        return prot_h, lig_h

In [ ]:
# ============================================================
# 模型架构 -- ToyDynamicBindScore (6 个输出头)
# ============================================================

class ToyDynamicBindScore(nn.Module):
    """
    DynamicBind 分数模型 -- 6 个输出头。

    与 DiffDock-Pocket 的架构对比:
      DiffDock:        tr_head + rot_head + tor_head                     (3 个头)
      DiffDock-Pocket: tr_head + rot_head + tor_head + sc_tor_head       (4 个头)
      DynamicBind:     tr_head + rot_head + tor_head                     (配体 3 个头)
                     + res_tr_head + res_rot_head + res_chi_head         (残基 3 个头)

    配体的 3 个头预测全局分数 (整个配体的平移/旋转/扭转)，
    残基的 3 个头预测逐残基分数 (每个残基独立的平移/旋转/chi角)。

    DynamicBind 原始实现中，残基的分数头使用 direction + magnitude 参数化:
      预测 = 方向向量 * magnitude_mlp(||方向向量||, sigma_embedding)
    这里简化为直接 MLP 预测。
    """
    def __init__(self, lig_dim=ATOM_FEAT_DIM, res_dim=RES_FEAT_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()

        # ---- 时间步嵌入 ----
        self.time_embed = SinusoidalEmbedding(hidden_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # ---- 蛋白残基嵌入 (残基级别: 21 -> 128) ----
        self.prot_embed = nn.Linear(res_dim, hidden_dim)

        # ---- 配体原子嵌入 (原子级别: 10 -> 128) ----
        self.lig_embed = nn.Linear(lig_dim, hidden_dim)

        # ---- 交互层 x 4 ----
        self.interaction_layers = nn.ModuleList([
            InteractionLayer(hidden_dim) for _ in range(N_INTERACTION_LAYERS)
        ])

        # ---- 配体的 3 个全局分数头 (与 DiffDock-Pocket 相同) ----
        self.tr_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3),
        )
        self.rot_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3),
        )
        self.tor_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

        # ---- DynamicBind 新增: 残基的 3 个逐残基分数头 ----
        # 残基平移分数: 每个残基预测 3D 平移方向
        self.res_tr_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 3),
        )
        # 残基旋转分数: 每个残基预测轴角旋转 (3D)
        self.res_rot_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 3),
        )
        # 残基侧链 chi 分数: 每个残基预测 chi1 扭转分数 (标量)
        self.res_chi_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, lig_feats, res_feats, edge_index, edge_dist, t):
        """
        参数:
          lig_feats  : (N_l, 10)   配体原子特征
          res_feats  : (N_r, 21)   蛋白残基特征
          edge_index : (2, E)      交互边 [0]=配体, [1]=蛋白
          edge_dist  : (E, 1)      边距离
          t          : (1,)        归一化时间步

        返回 6 个分数:
          score_tr      : (3,)       配体平移分数
          score_rot     : (3,)       配体旋转分数
          score_tor     : (1,)       配体扭转分数
          score_res_tr  : (N_r, 3)   残基平移分数
          score_res_rot : (N_r, 3)   残基旋转分数
          score_res_chi : (N_r, 1)   残基 chi 分数
        """
        # ---- 时间步条件 ----
        time_h = self.time_mlp(self.time_embed(t))   # (1, hidden)

        # ---- 节点嵌入 + 时间步注入 ----
        lig_h = self.lig_embed(lig_feats) + time_h
        prot_h = self.prot_embed(res_feats) + time_h

        # ---- 4 层交互 ----
        for layer in self.interaction_layers:
            prot_h, lig_h = layer(prot_h, lig_h, edge_index, edge_dist)

        # ---- 全局聚合 (mean + max) ----
        lig_mean = lig_h.mean(dim=0, keepdim=True)
        lig_max = lig_h.max(dim=0, keepdim=True).values
        lig_global = torch.cat([lig_mean, lig_max], dim=-1)

        prot_mean = prot_h.mean(dim=0, keepdim=True)
        prot_max = prot_h.max(dim=0, keepdim=True).values
        prot_global = torch.cat([prot_mean, prot_max], dim=-1)

        # ---- 配体的 3 个全局分数 (与 DiffDock-Pocket 相同) ----
        joint_global = lig_global + prot_global
        score_tr = self.tr_head(joint_global).squeeze(0)
        score_rot = self.rot_head(joint_global).squeeze(0)
        score_tor = self.tor_head(joint_global).squeeze(0)

        # ---- 残基的 3 个逐残基分数 (DynamicBind 新增) ----
        # 每个残基独立预测，使用残基级别的隐表示 prot_h
        score_res_tr = self.res_tr_head(prot_h)              # (N_r, 3)
        score_res_rot = self.res_rot_head(prot_h)            # (N_r, 3)
        score_res_chi = self.res_chi_head(prot_h)            # (N_r, 1)

        return score_tr, score_rot, score_tor, score_res_tr, score_res_rot, score_res_chi


# ---- 实例化模型 ----
model = ToyDynamicBindScore(
    lig_dim=ATOM_FEAT_DIM, res_dim=RES_FEAT_DIM, hidden_dim=HIDDEN_DIM
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"模型参数量: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")

# 显示模型结构摘要
head_info = pd.DataFrame({
    '输出头': ['tr_head (配体平移)', 'rot_head (配体旋转)', 'tor_head (配体扭转)',
              'res_tr_head (残基平移)', 'res_rot_head (残基旋转)', 'res_chi_head (残基chi)'],
    '输出维度': ['(3,) 全局', '(3,) 全局', '(1,) 全局',
              '(N_r, 3) 逐残基', '(N_r, 3) 逐残基', '(N_r, 1) 逐残基'],
    '来源': ['DiffDock', 'DiffDock', 'DiffDock',
            'DynamicBind 新增', 'DynamicBind 新增', 'DynamicBind 新增']
})
display(head_info)

## 5. 训练

训练过程使用 **6-way 分数匹配** (score matching) 损失:

1. 随机采样时间步 $t \in (0, 1]$
2. 对配体施加 3-way 噪声 (平移 + 旋转 + 扭转)
3. 对残基施加 3-way 噪声 (平移 + 旋转 + chi角) -- DynamicBind 新增
4. 模型预测 6 个分数，与真实分数进行匹配

分数匹配目标: $\text{score} = -\frac{\text{noise}}{\sigma^2}$

残基损失使用较小权重 (0.5)，因为蛋白运动幅度小于配体运动幅度。

In [ ]:
# ============================================================
# 训练循环 -- 6-way 分数匹配
# ============================================================

print(f"开始训练 {N_EPOCHS} 轮 (6-way score matching)...\n")

epoch_losses = []     # 记录每轮平均损失 (用于绘图)

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []
    loss_components = {'tr': [], 'rot': [], 'tor': [],
                       'res_tr': [], 'res_rot': [], 'res_chi': []}

    for item in train_loader:
        lig_feats = item['lig_feats']
        lig_coords = item['lig_coords']
        res_feats = item['res_feats']
        ca_coords = item['ca_coords']
        n_coords = item['n_coords']
        c_coords = item['c_coords']
        chi1_angles = item['chi1_angles']

        # ---- 采样随机时间步 t in (0, 1] ----
        t = np.random.uniform(0.01, 1.0)

        # ---- 1. 对配体施加 3-way 噪声 ----
        noisy_lig, noise_tr, noise_rot, noise_tor = apply_noise_to_ligand(lig_coords, t)

        # ---- 2. 对残基施加 3-way 噪声 (DynamicBind 新增!) ----
        noisy_ca, noisy_n, noisy_c, noisy_chi1, \
            noise_res_tr, noise_res_rot, noise_res_chi = \
            apply_noise_to_residues(ca_coords, n_coords, c_coords, chi1_angles, t)

        # ---- 3. 构建交互图 (使用加噪坐标) ----
        dist_mat = distance_matrix(noisy_lig, noisy_ca)
        lig_idx, prot_idx = np.where(dist_mat < DISTANCE_CUTOFF)
        if len(lig_idx) == 0:
            continue

        edge_index = np.stack([lig_idx, prot_idx], axis=0)
        edge_dist = dist_mat[lig_idx, prot_idx].reshape(-1, 1)

        # ---- 转换为张量 ----
        lig_f = torch.FloatTensor(lig_feats).to(DEVICE)
        res_f = torch.FloatTensor(res_feats).to(DEVICE)
        ei = torch.LongTensor(edge_index).to(DEVICE)
        ed = torch.FloatTensor(edge_dist).to(DEVICE)
        t_tensor = torch.FloatTensor([t]).to(DEVICE)

        # ---- 前向传播: 预测 6 个分数 ----
        score_tr, score_rot, score_tor, \
            score_res_tr, score_res_rot, score_res_chi = \
            model(lig_f, res_f, ei, ed, t_tensor)

        # ---- 6-way 分数匹配损失 ----
        # 分数匹配目标: score = -noise / sigma^2
        sigma_tr, sigma_rot, sigma_tor, \
            sigma_res_tr, sigma_res_rot, sigma_res_chi = t_to_sigma_6way(t)

        # 配体平移损失
        target_tr = torch.FloatTensor(-noise_tr / (sigma_tr ** 2 + 1e-8)).to(DEVICE)
        loss_tr = ((score_tr - target_tr) ** 2).sum()

        # 配体旋转损失
        target_rot = torch.FloatTensor(-noise_rot / (sigma_rot ** 2 + 1e-8)).to(DEVICE)
        loss_rot = ((score_rot - target_rot) ** 2).sum()

        # 配体扭转损失
        target_tor = torch.FloatTensor([-noise_tor / (sigma_tor ** 2 + 1e-8)]).to(DEVICE)
        loss_tor = ((score_tor - target_tor) ** 2).sum()

        # 残基平移损失 -- DynamicBind 新增!
        target_res_tr = torch.FloatTensor(
            -noise_res_tr / (sigma_res_tr ** 2 + 1e-8)
        ).to(DEVICE)
        loss_res_tr = ((score_res_tr - target_res_tr) ** 2).mean()

        # 残基旋转损失 -- DynamicBind 新增!
        target_res_rot = torch.FloatTensor(
            -noise_res_rot / (sigma_res_rot ** 2 + 1e-8)
        ).to(DEVICE)
        loss_res_rot = ((score_res_rot - target_res_rot) ** 2).mean()

        # 残基 chi 损失 (与 DiffDock-Pocket 的 sc_tor 对应)
        target_res_chi = torch.FloatTensor(
            -noise_res_chi / (sigma_res_chi ** 2 + 1e-8)
        ).reshape(-1, 1).to(DEVICE)
        loss_res_chi = ((score_res_chi - target_res_chi) ** 2).mean()

        # ---- 总损失: 6 个分量的加权和 ----
        # 残基损失使用较小权重，因为蛋白运动幅度小
        total_loss = (loss_tr + loss_rot + loss_tor
                      + 0.5 * loss_res_tr + 0.5 * loss_res_rot + 0.5 * loss_res_chi)

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        train_losses.append(total_loss.item())
        loss_components['tr'].append(loss_tr.item())
        loss_components['rot'].append(loss_rot.item())
        loss_components['tor'].append(loss_tor.item())
        loss_components['res_tr'].append(loss_res_tr.item())
        loss_components['res_rot'].append(loss_res_rot.item())
        loss_components['res_chi'].append(loss_res_chi.item())

    avg_total = np.mean(train_losses) if train_losses else float('nan')
    epoch_losses.append(avg_total)

    # ---- 每 20 轮打印一次 ----
    if epoch % 20 == 0 or epoch == 1:
        avg_tr = np.mean(loss_components['tr']) if loss_components['tr'] else 0
        avg_rot = np.mean(loss_components['rot']) if loss_components['rot'] else 0
        avg_tor = np.mean(loss_components['tor']) if loss_components['tor'] else 0
        avg_res_tr = np.mean(loss_components['res_tr']) if loss_components['res_tr'] else 0
        avg_res_rot = np.mean(loss_components['res_rot']) if loss_components['res_rot'] else 0
        avg_res_chi = np.mean(loss_components['res_chi']) if loss_components['res_chi'] else 0
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  Total: {avg_total:.4f}  "
              f"(tr={avg_tr:.3f}, rot={avg_rot:.3f}, tor={avg_tor:.3f}, "
              f"res_tr={avg_res_tr:.3f}, res_rot={avg_res_rot:.3f}, res_chi={avg_res_chi:.3f})")

print("\n训练完成!")

## 6. 评估与可视化

在测试集上进行 **6-way 反向扩散对接**:

1. 从纯噪声开始 ($t=1.0$)
2. 逐步去噪: $t=1.0 \to t=0$，共 20 步
3. 每步使用模型预测的 6 个分数进行朗之万动力学更新

评估指标:
- **配体 RMSD**: 预测配体位姿与真实位姿的均方根偏差
- **残基 CA RMSD**: 预测的 CA 原子位置与真实位置的偏差 (DynamicBind 新增)
- **残基旋转 RMSD**: N-CA-C 局部坐标系的角度偏差 (DynamicBind 新增)
- **侧链 chi1 RMSD**: 预测的侧链扭转角与真实值的偏差

In [ ]:
# ============================================================
# 测试与评估 -- 6-way 反向扩散对接
# ============================================================

print("在测试集上评估 (6-way 反向扩散对接)...")

model.eval()
lig_rmsds = []
res_tr_rmsds = []
res_rot_rmsds = []
chi_rmsds = []
pdbids_list = []

# ---- 扩散采样参数 ----
N_STEPS = 20           # 反向扩散步数
DT = 1.0 / N_STEPS

with torch.no_grad():
    for item in test_loader:
        lig_feats = item['lig_feats']
        lig_coords_true = item['lig_coords']
        res_feats = item['res_feats']
        ca_coords_true = item['ca_coords']
        n_coords_true = item['n_coords']
        c_coords_true = item['c_coords']
        chi1_true = item['chi1_angles']
        pdbid = item['pdbid']

        # ---- 初始化: 从纯噪声开始 (t=1.0) ----
        noisy_lig, _, _, _ = apply_noise_to_ligand(lig_coords_true, 1.0)
        noisy_ca, noisy_n, noisy_c, noisy_chi1, _, _, _ = \
            apply_noise_to_residues(
                ca_coords_true, n_coords_true, c_coords_true, chi1_true, 1.0
            )

        # ---- 反向扩散: t=1.0 -> t=0 ----
        for step in range(N_STEPS):
            t = 1.0 - step * DT
            sigma_tr, sigma_rot, sigma_tor, \
                sigma_res_tr, sigma_res_rot, sigma_res_chi = t_to_sigma_6way(t)

            # 构建交互图
            dist_mat = distance_matrix(noisy_lig, noisy_ca)
            lig_idx, prot_idx = np.where(dist_mat < DISTANCE_CUTOFF)
            if len(lig_idx) == 0:
                break

            edge_index = np.stack([lig_idx, prot_idx], axis=0)
            edge_dist = dist_mat[lig_idx, prot_idx].reshape(-1, 1)

            lig_f = torch.FloatTensor(lig_feats).to(DEVICE)
            res_f = torch.FloatTensor(res_feats).to(DEVICE)
            ei = torch.LongTensor(edge_index).to(DEVICE)
            ed = torch.FloatTensor(edge_dist).to(DEVICE)
            t_tensor = torch.FloatTensor([t]).to(DEVICE)

            score_tr, score_rot, score_tor, \
                score_res_tr, score_res_rot, score_res_chi = \
                model(lig_f, res_f, ei, ed, t_tensor)

            # ---- 配体去噪更新 (朗之万动力学) ----
            step_size = DT

            # 1. 配体平移更新
            noisy_lig = noisy_lig + score_tr.cpu().numpy() * sigma_tr ** 2 * step_size

            # 2. 配体旋转更新
            rot_update = score_rot.cpu().numpy() * sigma_rot ** 2 * step_size
            angle = np.linalg.norm(rot_update)
            if angle > 1e-6:
                axis = rot_update / angle
                center = noisy_lig.mean(axis=0)
                cos_a, sin_a = np.cos(angle), np.sin(angle)
                centered = noisy_lig - center
                cross = np.cross(axis, centered)
                dot = np.dot(centered, axis).reshape(-1, 1)
                noisy_lig = (centered * cos_a + cross * sin_a
                             + axis * dot * (1 - cos_a) + center)

            # 3. 配体扭转更新
            tor_update = score_tor.cpu().numpy().item() * sigma_tor ** 2 * step_size
            center = noisy_lig.mean(axis=0)
            radial = noisy_lig - center
            radial_norms = np.linalg.norm(radial, axis=1, keepdims=True)
            radial_norms = np.clip(radial_norms, 1e-6, None)
            noisy_lig = noisy_lig + (radial / radial_norms) * tor_update * 0.1

            # ---- 残基去噪更新 (DynamicBind 新增!) ----
            # 原始代码中的 adaptive dynamics: 仅在配体接近时更新残基
            # 这里简化为始终更新，使用衰减因子
            decay = 1.0 / (N_STEPS - step + N_STEPS * 0.25)

            # 4. 残基平移更新
            res_tr_update = score_res_tr.cpu().numpy() * decay
            res_tr_update = np.clip(res_tr_update, -20, 20)
            noisy_ca = noisy_ca + res_tr_update
            noisy_n = noisy_n + res_tr_update
            noisy_c = noisy_c + res_tr_update

            # 5. 残基旋转更新 (绕 CA 旋转 N 和 C)
            res_rot_update = score_res_rot.cpu().numpy() * decay
            res_rot_update = np.clip(res_rot_update, -20, 20)
            for i in range(len(noisy_ca)):
                rot_angle = np.linalg.norm(res_rot_update[i])
                if rot_angle > 1e-6:
                    rot_mat = axis_angle_to_matrix(res_rot_update[i])
                    noisy_n[i] = rot_mat @ (noisy_n[i] - noisy_ca[i]) + noisy_ca[i]
                    noisy_c[i] = rot_mat @ (noisy_c[i] - noisy_ca[i]) + noisy_ca[i]

            # 6. 残基 chi 角更新
            chi_update = score_res_chi.cpu().numpy().flatten() * decay
            noisy_chi1 = noisy_chi1 + chi_update
            noisy_chi1 = np.arctan2(np.sin(noisy_chi1), np.cos(noisy_chi1))

        # ---- 计算配体 RMSD (质心对齐后) ----
        pred_center = noisy_lig.mean(axis=0)
        true_center = lig_coords_true.mean(axis=0)
        pred_aligned = noisy_lig - pred_center + true_center
        lig_rmsd = compute_rmsd(pred_aligned, lig_coords_true)
        lig_rmsds.append(lig_rmsd)

        # ---- 计算残基 CA 平移 RMSD ----
        res_tr_rmsd = compute_rmsd(noisy_ca, ca_coords_true)
        res_tr_rmsds.append(res_tr_rmsd)

        # ---- 计算残基旋转 RMSD (N-CA-C 局部坐标系偏差) ----
        # 用 N-CA 和 C-CA 方向向量的角度偏差来衡量
        true_n_dir = n_coords_true - ca_coords_true
        pred_n_dir = noisy_n - noisy_ca
        true_n_norm = np.linalg.norm(true_n_dir, axis=1, keepdims=True)
        pred_n_norm = np.linalg.norm(pred_n_dir, axis=1, keepdims=True)
        true_n_dir = true_n_dir / np.clip(true_n_norm, 1e-6, None)
        pred_n_dir = pred_n_dir / np.clip(pred_n_norm, 1e-6, None)
        cos_sim = np.clip(np.sum(true_n_dir * pred_n_dir, axis=1), -1, 1)
        angle_err = np.arccos(cos_sim)
        res_rot_rmsd = np.sqrt(np.mean(angle_err ** 2))
        res_rot_rmsds.append(res_rot_rmsd)

        # ---- 计算侧链 chi1 角度 RMSD (考虑周期性) ----
        chi1_diff = noisy_chi1 - chi1_true
        chi1_diff = np.arctan2(np.sin(chi1_diff), np.cos(chi1_diff))
        chi_rmsd = np.sqrt(np.mean(chi1_diff ** 2))
        chi_rmsds.append(chi_rmsd)

        pdbids_list.append(pdbid)

lig_rmsds = np.array(lig_rmsds)
res_tr_rmsds = np.array(res_tr_rmsds)
res_rot_rmsds = np.array(res_rot_rmsds)
chi_rmsds = np.array(chi_rmsds)

print(f"\n评估完成! 共 {len(lig_rmsds)} 个测试样本")

In [ ]:
# ============================================================
# 评估结果展示与可视化
# ============================================================

# ---- 结果统计表格 ----
results_df = pd.DataFrame({
    '指标': [
        '样本数',
        '配体 RMSD (mean)',
        '配体 RMSD (median)',
        '配体 RMSD < 2A',
        '配体 RMSD < 5A',
        '残基 CA RMSD (mean)',
        '残基 CA RMSD (median)',
        '残基旋转 RMSD (mean)',
        '侧链 chi1 RMSD (mean)',
    ],
    '值': [
        str(len(lig_rmsds)),
        f'{lig_rmsds.mean():.4f} A',
        f'{np.median(lig_rmsds):.4f} A',
        f'{(lig_rmsds < 2.0).sum()}/{len(lig_rmsds)} ({100.0*(lig_rmsds < 2.0).mean():.1f}%)',
        f'{(lig_rmsds < 5.0).sum()}/{len(lig_rmsds)} ({100.0*(lig_rmsds < 5.0).mean():.1f}%)',
        f'{res_tr_rmsds.mean():.4f} A',
        f'{np.median(res_tr_rmsds):.4f} A',
        f'{np.degrees(res_rot_rmsds.mean()):.2f} deg',
        f'{np.degrees(chi_rmsds.mean()):.2f} deg',
    ],
    '类别': [
        '',
        '配体', '配体', '配体', '配体',
        '残基 (DynamicBind 新增)', '残基 (DynamicBind 新增)',
        '残基 (DynamicBind 新增)', '残基 (DynamicBind 新增)',
    ]
})
display(results_df)

# ---- 可视化: 3 个子图 ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 子图 1: 配体 RMSD 直方图
ax1 = axes[0]
ax1.hist(lig_rmsds, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
ax1.axvline(x=2.0, color='red', linestyle='--', linewidth=1.5, label='2 A threshold')
ax1.axvline(x=5.0, color='orange', linestyle='--', linewidth=1.5, label='5 A threshold')
ax1.set_xlabel('Ligand RMSD (A)', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title(f'Ligand RMSD Distribution\n'
              f'Mean={lig_rmsds.mean():.2f} A, Median={np.median(lig_rmsds):.2f} A',
              fontsize=11)
ax1.legend(fontsize=10)

# 子图 2: 残基 CA RMSD vs 配体 RMSD 散点图
ax2 = axes[1]
ax2.scatter(lig_rmsds, res_tr_rmsds, alpha=0.6, edgecolors='k',
            linewidths=0.5, s=40, color='coral')
ax2.set_xlabel('Ligand RMSD (A)', fontsize=12)
ax2.set_ylabel('Residue CA RMSD (A)', fontsize=12)
ax2.set_title(f'DynamicBind: 6-way Diffusion\n'
              f'Residue CA RMSD Mean={res_tr_rmsds.mean():.2f} A',
              fontsize=11)
ax2.annotate('6-way diffusion:\nlig tr/rot/tor\n+ res tr/rot/chi',
             xy=(0.95, 0.95), xycoords='axes fraction',
             ha='right', va='top', fontsize=9,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

# 子图 3: 训练损失曲线
ax3 = axes[2]
ax3.plot(range(1, len(epoch_losses) + 1), epoch_losses, 'b-', linewidth=1.0, alpha=0.8)
ax3.set_xlabel('Epoch', fontsize=12)
ax3.set_ylabel('Training Loss', fontsize=12)
ax3.set_title('Training Loss Curve\n(6-way Score Matching)', fontsize=11)
ax3.set_yscale('log')

plt.tight_layout()
plt.show()

print("可视化完成!")

## 总结

本教程实现了简化版的 DynamicBind 模型，展示了其核心思想：**6-way 扩散**同时建模配体和蛋白残基的完整运动自由度。

### 从 DiffDock 到 DynamicBind 的演化

| 特性 | DiffDock | DiffDock-Pocket | DynamicBind |
|------|----------|-----------------|-------------|
| 配体平移 | Yes | Yes | Yes |
| 配体旋转 | Yes | Yes | Yes |
| 配体扭转 | Yes | Yes | Yes |
| 侧链 chi 角 | - | Yes | Yes |
| 残基平移 | - | - | **Yes** |
| 残基旋转 | - | - | **Yes** |
| 噪声调度 | Exponential | Exponential | **Exp (配体) + Power-law (残基)** |
| 蛋白构象变化 | 不建模 | 仅侧链 | **全残基运动** |

### 关键设计选择

1. **差异化噪声调度**: 蛋白残基使用 power-law 调度而非指数调度，噪声范围更小 (最大 1A vs 10A)，反映蛋白骨架运动的物理约束
2. **逐残基分数预测**: 每个残基独立预测平移/旋转/chi 分数，而非全局预测
3. **局部坐标系**: 使用 N-CA-C 三点定义残基的局部坐标系，确保旋转表示的物理合理性

### 注意事项

- 本教程为教学用简化版本，使用 CASF-2016 数据集的 285 个复合物
- 完整版 DynamicBind 还包含 SE(3) 等变网络、多置信度排序等高级特性
- 由于数据量和模型规模的限制，本教程的对接精度远低于原始论文报告的结果